In [1]:
import torch
import wandb

from datasets.mnist import MNISTSampler
from models.config import load_config
from models.flow import FlowModel
from training.path import GaussianConditionalProbabilityPath, LinearAlpha, LinearBeta
from training.trainer import FlowTrainer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device", device)

CONFIG = "configs/mnist.yaml"
cfg = load_config(CONFIG)
flow = FlowModel.from_config(CONFIG).to(device)

xt = torch.randn(
    cfg["test"]["batch_size"],
    cfg["unet"]["in_channels"],
    cfg["test"]["image_size"],
    cfg["test"]["image_size"],
    device=device,
)
t = torch.rand(xt.shape[0], device=device)
print("u(x, t)", tuple(flow(xt, t).shape))

device mps
u(x, t) (2, 1, 32, 32)


In [2]:
def make_path(split: str) -> GaussianConditionalProbabilityPath:
    return GaussianConditionalProbabilityPath(
        p_data=MNISTSampler(split=split),
        p_simple_shape=[1, 32, 32],
        alpha=LinearAlpha(),
        beta=LinearBeta(),
    ).to(device)

train_path = make_path("train")
val_path = make_path("val")

trainer = FlowTrainer(path=train_path, model=flow, val_path=val_path)
run = wandb.init(
    project="mnist-flow-matching",
    config={
        "num_steps": 5000,
        "batch_size": 64,
        "learning_rate": 1e-3,
        "model": cfg,
    },
)
try:
    history = trainer.train(
        num_steps=5000,
        device=device,
        lr=1e-3,
        batch_size=64,
        ckpt_path="checkpoints/mnist_flow.pt",
        checkpoint_every=50,
        val_every=50,
        val_batches=8,
        plot_every=50,
        n_plot_images=10,
        n_plot_steps=10,
        samples_dir="samples",
        show_plots=False,
        wandb_run=run,
    )
finally:
    run.finish()

cache MNIST val: 100%|██████████| 5000/5000 [00:01<00:00, 2788.43it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/sairiteshthela/.netrc.
wandb: Currently logged in as: private-wandb-account (private-wandb-account) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training model with size: 50.859 MiB


Step 5000, train: 0.136, val: 0.141: 100%|██████████| 5000/5000 [1:10:06<00:00,  1.19it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


saved checkpoints/mnist_flow.pt


loss/train,█▅▇█▇▅▃▅▆▃▄▄▄▄▂▄▄▁▃▃▅▂▂▄▃▃▃▃▃▆▅▂▂▃▃▂▃▃▄▂
loss/val,█▇▆▅▅▃▄▃▃▃▃▃▂▂▂▂▂▂▁▃▂▂▂▂▂▁▃▂▂▂▂▁▂▂▁▂▂▂▂▂
loss/train,0.13584
loss/val,0.14128
